## 1. Get the repository (skip if a full checkout already exists)

We recommend running `git clone https://github.com/ShellWee/inspection-demo.git` in a terminal first, then opening this notebook. To update an existing checkout, run `git pull --ff-only` in the repository directory.

If you only opened this `.ipynb` in Colab, you can provide the repository HTTPS URL in the next cell to clone it. Accessing a private repository requires Git credentials to be configured in advance; do not put a token in the URL, cell, or output. Leave it empty to skip cloning.


In [ ]:
from pathlib import Path
import os
import subprocess
from urllib.parse import urlsplit

GIT_URL = ""  # Set to https://github.com/ShellWee/inspection-demo.git only if no checkout exists. Never include a token.
CHECKOUT_DIR = Path.cwd() / "inspection-demo-hf"

if GIT_URL:
    parsed_url = urlsplit(GIT_URL)
    if (parsed_url.scheme != "https" or not parsed_url.hostname
            or parsed_url.username or parsed_url.password
            or parsed_url.query or parsed_url.fragment):
        raise ValueError("Use a repository HTTPS URL without credentials.")
    if CHECKOUT_DIR.exists():
        print("Checkout directory exists; update it with git pull --ff-only if needed.")
    else:
        git_env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
        subprocess.run(["git", "clone", GIT_URL, str(CHECKOUT_DIR)],
                       check=True, env=git_env)
else:
    print("Using the existing checkout.")


## 2. Configuration

`ASSETS_DIR` defaults to `assets` inside the repository, but you can change it to the path of an existing asset bundle. If assets are missing, set `DOWNLOAD_ASSETS` to `True` to download a pinned version of the private dataset using your existing Hugging Face login. Private asset access permissions are independent of Git repository permissions.

`DEVICE="cuda"` only works in environments with a compatible NVIDIA GPU; when CUDA is selected, the launcher validates the device. The notebook kernel does not need Torch or FastAPI from the app environment installed.


In [ ]:
from pathlib import Path
import os
import sys

if sys.version_info < (3, 10):
    raise RuntimeError("Use an approved Python 3.10+ notebook kernel.")

def find_repo(start):
    start = Path(start).resolve()
    candidates = [start, *start.parents,
                  start / "inspection-demo-hf", start / "inspection-demo"]
    for candidate in candidates:
        if ((candidate / "scripts" / "demo_launcher.py").is_file()
                and (candidate / "pyproject.toml").is_file()):
            return candidate
    raise RuntimeError("Clone the complete repository, then open this notebook from that checkout.")

REPO_ROOT = find_repo(Path.cwd())
ASSETS_DIR = REPO_ROOT / "assets"  # Or Path("/path/to/existing/asset-bundle")
STATE_DIR = REPO_ROOT / ".demo-state"
DEVICE = "cpu"                   # "cpu" or "cuda"
DOWNLOAD_ASSETS = False           # Explicit opt-in to private HF download
PORT = 7860
MODEL_ID = "gpt-4.1"              # Also: "gpt-5", "gpt-5.6-luna"
IN_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
BASE_URL = f"http://127.0.0.1:{PORT}"

print("Repository:", REPO_ROOT)
print("Assets:", ASSETS_DIR)
print("Mode:", "Colab HTTP results" if IN_COLAB else "Local browser + HTTP results")


## 3. Set up the application environment

First make sure the current kernel can run `uv`. If you need the notebook to install it, set `INSTALL_UV` to `True` in the next cell; it installs only a pinned version of `uv` into the current kernel environment. Setup then uses `uv.lock` to create the app's Python 3.12 environment. The repository already includes a built frontend, so startup does not require Node.js.


In [ ]:
import subprocess
import sys

INSTALL_UV = False
if INSTALL_UV:
    subprocess.run([sys.executable, "-m", "pip", "install", "uv==0.12.9"], check=True)


In [ ]:
import subprocess
import sys

def run_launcher(*arguments):
    return subprocess.run(
        [sys.executable, str(REPO_ROOT / "scripts" / "demo_launcher.py"), *map(str, arguments)],
        cwd=REPO_ROOT, check=True,
    )

run_launcher("setup", "--device", DEVICE)


## 4. Verify or download assets

Keep `DOWNLOAD_ASSETS=False` when you already have a complete asset bundle. If you need to download, first run `hf auth login` in the current runtime (read-only permission with approved access to the private dataset), then set it to `True`. Downloads use a pinned dataset version; this cell does not accept a token.


In [ ]:
asset_arguments = ["assets", "--assets-dir", ASSETS_DIR]
if DOWNLOAD_ASSETS:
    asset_arguments.append("--download")
run_launcher(*asset_arguments)


## 5. Start and check

The service listens only on `127.0.0.1`. The launcher waits until the service is ready and stores process information in `STATE_DIR`, so closing the browser cell does not stop the service; the final cell can stop it explicitly. After restarting the notebook kernel, rerun the configuration and function-definition cells to check/stop the service using the same state directory.


In [ ]:
run_launcher("start", "--assets-dir", ASSETS_DIR, "--device", DEVICE,
             "--port", PORT, "--state-dir", STATE_DIR)
if IN_COLAB:
    print("Ready. Use the query cell below; no public browser URL is created.")
else:
    print(f"Open on this computer: {BASE_URL}")


In [ ]:
run_launcher("status", "--state-dir", STATE_DIR)


## 6. Optional: submit a query in the notebook

Before running a paid query, set `RUN_QUERY` to `True`. Enter your original question in English, then provide an OpenAI API key through hidden input. The client submits once, then polls until `completed`, `abstained`, `failed`, or `cancelled`.

The key is used only for the request and is not written to environment variables, command-line arguments, or files; keys in responses are masked, and the notebook key variable is cleared afterward. Timeouts or interruptions do not cancel an already submitted job, so avoid immediate retries that could create duplicate calls. Results include your query and building information, so clear all outputs before sharing or submitting the notebook.


In [ ]:
import getpass
import importlib.util
import json
import warnings

RUN_QUERY = False
if RUN_QUERY:
    spec = importlib.util.spec_from_file_location(
        "inspection_notebook_client", REPO_ROOT / "scripts" / "notebook_client.py")
    notebook_client = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(notebook_client)
    query = input("English inspection query (3–2000 characters): ")
    api_key = None
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("error", getpass.GetPassWarning)
            api_key = getpass.getpass("OpenAI API key (hidden): ")
        run_result = notebook_client.run_grounding(
            base_url=BASE_URL, query=query, model_id=MODEL_ID, api_key=api_key)
        print(json.dumps(run_result, ensure_ascii=False, indent=2))
    except notebook_client.NotebookClientError as error:
        print(str(error))
    except getpass.GetPassWarning:
        print("Hidden input is unavailable. Use a notebook frontend that supports getpass.")
    except KeyboardInterrupt:
        print("Client interrupted; an accepted backend run may still be running.")
    except Exception as error:
        print(f"Query stopped ({type(error).__name__}); no diagnostic body is displayed.")
    finally:
        api_key = ""
        del api_key
else:
    print("No API call made. Set RUN_QUERY=True when ready.")


## 7. Stop the service

When you need to stop the service, set `STOP_SERVER` to `True` and run this cell. It is skipped by default to avoid shutting down the service immediately during “Run All”. Local runtime data in the state directory is managed by the launcher; after stopping, you can run the start cell again.


In [ ]:
STOP_SERVER = False
if STOP_SERVER:
    run_launcher("stop", "--state-dir", STATE_DIR)
else:
    print("Server left running. Set STOP_SERVER=True to stop it.")
